In [3]:
import pandas as pd
import os
import shutil
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPTokenizer, CLIPTextModel
from openai import OpenAI
import json
import re
import anthropic
import torch
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [5]:
Artist_name = "Pierre-Auguste Renoir"

In [6]:
artnet_2024_Renoir = artnet_2024[artnet_2024["artist"] == Artist_name].reset_index(drop=True)

In [7]:
source_folder = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images"
destination_folder = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\artwork_label"

In [9]:
artnet_2024_Renoir.columns

Index(['artwork id', 'artist id', 'first', 'last', 'nationality', 'year born',
       'year died', 'title', 'workyear modifier', 'workyear from',
       'workyear to', 'est lo usd', 'est hi usd', 'sale price usd', 'artist',
       'price'],
      dtype='object')

In [8]:
artnet_2024_Renoir

,artwork id,artist id,first,last,nationality,year born,year died,title,workyear modifier,workyear from,workyear to,est lo usd,est hi usd,sale price usd,artist,price
0,41,14139,Pierre-Auguste,Renoir,French,1841,1919,Baigneuse,NaN,1888,1888.0,1.000000e+07,1.500000e+07,2.090250e+07,Pierre-Auguste Renoir,4.085286e+07
1,62,14139,Pierre-Auguste,Renoir,French,1841,1919,"Femme au peplum rouge, tête, bras",NaN,1895,NaN,8.695652e+04,1.217391e+05,1.920000e+05,Pierre-Auguste Renoir,2.987514e+05
2,149,14139,Pierre-Auguste,Renoir,French,1841,1919,Saule au bord d'une mare,CIRCA,1874,NaN,7.906674e+04,1.143112e+05,1.060334e+05,Pierre-Auguste Renoir,1.362007e+05
3,628,14139,Pierre-Auguste,Renoir,French,1841,1919,Compotier de fruits,CIRCA,1890,NaN,2.500000e+05,3.500000e+05,5.065000e+05,Pierre-Auguste Renoir,7.405878e+05
4,1184,14139,Pierre-Auguste,Renoir,French,1841,1919,Paysage,NaN,1917,NaN,8.150599e+04,1.191084e+05,9.526623e+04,Pierre-Auguste Renoir,1.367884e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2900,444330518,14139,Pierre-Auguste,Renoir,French,1841,1919,卡涅斯风景,NaN,1900,NaN,2.748952e+05,4.123428e+05,4.109683e+05,Pierre-Auguste Renoir,4.109683e+05
2901,444378044,14139,Pierre-Auguste,Renoir,French,1841,1919,Sur la plage à Bernval,CIRCA,1892,NaN,6.303183e+02,6.303183e+02,6.303183e+02,Pierre-Auguste Renoir,6.303183e+02
2902,444486964,14139,Pierre-Auguste,Renoir,French,1841,1919,Une Mère et Enfants,CIRCA,1912,NaN,6.000000e+02,9.000000e+02,3.250000e+02,Pierre-Auguste Renoir,3.250000e+02
2903,444878391,14139,Pierre-Auguste,Renoir,French,1841,1919,"Baigneuse debout, à mi-jambes",NaN,1910,NaN,4.760897e+02,4.760897e+02,4.760897e+02,Pierre-Auguste Renoir,4.760897e+02


In [16]:
for i in range(artnet_2024_Renoir.shape[0]):
    filename = f"{artnet_2024_Renoir.iloc[i]['artwork id']}.jpg"
    shutil.copy2(os.path.join(source_folder, filename),
                     os.path.join(destination_folder, filename))

In [9]:
artnet_2024_Renoir.to_csv(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Renoir_artworks.csv",index=False)

# JSON File

In [15]:
tasks = []

In [16]:
artnet_2024_Renoir = artnet_2024_Renoir.sort_values(by='workyear from', ascending=True)

In [17]:
for _, row in artnet_2024_Renoir.iterrows():
    task = {
        "data": {
            "image":       f"gs://artwork_labeling/artwork_label/{row['artwork id']}.jpg",
            "artist":      f"{row['first']} {row['last']}",
            "nationality": row['nationality'],
            "title":       row['title'],
            "year":        str(row['workyear from']) if pd.notna(row['workyear from']) else "Unknown"
        }
    }
    tasks.append(task)

In [18]:
# Save to JSON
with open('tasks.json', 'w', encoding='utf-8') as f:
    json.dump(tasks, f, indent=2, ensure_ascii=False)